# FA-KGD Multicoil-ACS + T-Convergence Sweep — Colab runner

Two experiments:
1. **R-sweep** at fixed T=20: R∈{4,8,12,16} × {oracle, multicoil_acs}
2. **T-convergence**: T∈{10,20,50,100} × R∈{4,8} with ΠGDM and FA-KGD only

Noise model: **freq-dependent** (σ²(r) = σ_base·(1 + β·(r/r_max)²), β=5).
White (isotropic) noise collapses FA-KGD's frequency-adaptive Kalman gain
to a scalar identical to ΠGDM's, so the whole method is a no-op there.

**Prereqs in `MyDrive/fastmri_artifacts/`:**
- `network-snapshot.pkl` (250 MB)
- `brain_12vols.tar` (~3.6 GB)

**Runtime → T4 GPU.** Wall time ~3–4 h total.

In [ ]:
# Cell 1 — verify GPU
!nvidia-smi | head -20

In [ ]:
# Cell 2 — clone main repo (or git pull if already there); ensure ADPS source tree
# (ADPS provides dnnlib + torch_utils, which the EDM checkpoint pickle references.)
import os
%cd /content
if os.path.isdir('/content/fastmri/.git'):
    %cd /content/fastmri
    !git fetch --quiet && git reset --hard origin/main
else:
    !rm -rf fastmri
    !git clone https://github.com/carlo-scr/fastmri.git
    %cd /content/fastmri
if not os.path.isdir('external/adps/dnnlib'):
    !rm -rf external/adps
    !git clone --depth 1 https://github.com/utcsilab/ambient-diffusion-mri.git external/adps
!ls external/adps/dnnlib external/adps/torch_utils | head -5
!git --no-pager log -1 --oneline

In [ ]:
# Cell 3 — install deps + preflight checks
# (Do NOT pin numpy here: Colab ships scikit-image built against numpy>=2.3,
# and downgrading numpy breaks its C extensions with ImportError on _center.)
!pip install -q h5py s3fs wandb pyyaml fastmri

# Preflight: ensure Colab has the updated reconstruct.py with gated M-step CLI.
import subprocess
help_text = subprocess.run(
    ['python', 'scripts/reconstruct.py', '--help'],
    check=True, capture_output=True, text=True,
).stdout
assert '--m_step_start_frac' in help_text, (
    'Colab cloned an older fastmri commit that does not include --m_step_start_frac. '
    'Commit + push scripts/reconstruct.py and src/samplers/fakgd.py locally, then re-run Cell 2.'
)
print('OK: gated M-step CLI present')


In [ ]:
# Cell 4 — mount Drive and stage artifacts
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
ART = '/content/drive/MyDrive/fastmri_artifacts'
assert os.path.exists(f'{ART}/network-snapshot.pkl'), 'checkpoint missing in Drive'
assert os.path.exists(f'{ART}/brain_12vols.tar'),    'tarball missing in Drive'

os.makedirs('checkpoints/edm/supervised_R=1', exist_ok=True)
shutil.copy(f'{ART}/network-snapshot.pkl', 'checkpoints/edm/supervised_R=1/network-snapshot.pkl')

# Tarball preserves data/multicoil_val/... paths, so extract from repo root
!tar xf "$ART/brain_12vols.tar" -C /content/fastmri

# Strip macOS AppleDouble junk (._* / .DS_Store) that BSD tar on macOS injects;
# h5py would otherwise try to open them and fail with 'file signature not found'.
!find data/multicoil_val -name '._*' -delete
!find data/multicoil_val -name '.DS_Store' -delete

# Sanity-check every file actually opens as HDF5
import h5py, glob
files = sorted(glob.glob('data/multicoil_val/*.h5'))
bad = []
for f in files:
    try:
        with h5py.File(f, 'r'): pass
    except Exception as e:
        bad.append((f, str(e)))
print(f'{len(files)} h5 files; {len(bad)} unreadable')
for f, e in bad: print('  BAD:', f, '→', e)
!ls -la data/multicoil_val/ | head -15
!ls -la checkpoints/edm/supervised_R=1/

In [ ]:
# Cell 10 — back up everything to Drive
import os, tarfile
ART = '/content/drive/MyDrive/fastmri_artifacts'  # self-contained
RSWEEP_TAG='fd_b5_smax10_clamp_v4'
TSWEEP_TAG='fd_b5_smax10_clamp_oracle_v4'
out_tar = f'{ART}/sweep_results_{RSWEEP_TAG}.tar'
with tarfile.open(out_tar, 'w') as tf:
    for d in (f'outputs/Rsweep_T20_{RSWEEP_TAG}', f'outputs/Tsweep_{TSWEEP_TAG}'):
        if os.path.exists(d):
            tf.add(d, arcname=os.path.basename(d))
            print(f'  + {d}')
        else:
            print(f'  - {d} (missing, skipping)')
print('Saved:', out_tar)
!ls -la "$ART" 2>/dev/null || ls -la /content/drive/MyDrive/fastmri_artifacts

## Paper table upgrades: bump all main-body tables from 5-slice pilot to 192-slice / 12-volume

Run these once on a Colab A100 to fully back the following tables:
- `tab:results` (Table 2) — DPS + ADPS + VarNet baselines on the 12-vol set
- `tab:scaling` (Table 5) — adds DPS column at brain R=4, T in {20,50,100}
- `tab:freq` (Table 6) — per-band k-space NMSE on 192 brain R=4 slices
- `tab:hparam` (Table 7) — 12-cell beta x alpha grid on 192 brain R=4 slices

Each cell skips runs whose results.json already exists, so the cells are restartable.


In [ ]:
# Cell U1 - Brain DPS + ADPS T-sweep on 192 slices, R in {4,8}, T in {20,50,100}.
# Provides DPS column for tab:scaling AND fills DPS/ADPS rows of tab:results
# at scale. Uses a separate output tag so existing Tsweep_fd_b5_smax10_clamp_oracle_v4
# (PiGDM+FA-KGD) results are NOT overwritten.
import os, subprocess, time
REPO   = '/content/fastmri'
DATA   = 'data/multicoil_val'
CKPT   = 'checkpoints/edm/supervised_R=1'
NSL    = 12 * 16  # 192
CF     = {4: 0.08, 8: 0.04}
TAG    = 'fd_b5_smax10_clamp_oracle_v4_dpsadps'
OUTROOT = f'outputs/Tsweep_{TAG}'
os.makedirs(f'{REPO}/{OUTROOT}', exist_ok=True)
for R in (4, 8):
    for T in (20, 50, 100):
        out = f'{OUTROOT}/R{R}_T{T}'
        if os.path.exists(f'{REPO}/{out}/results.json'):
            print('  skip (exists):', out); continue
        print(f'\n===== brain DPS+ADPS R={R} T={T} -> {out} =====')
        t0 = time.time()
        cmd = ['python','-u','scripts/reconstruct.py',
            '--mode','edm','--checkpoint_dir',CKPT,'--data_path',DATA,
            '--num_slices',str(NSL),'--whole_volume',
            '--acceleration',str(R),'--center_fraction',str(CF[R]),
            '--num_steps',str(T),'--schedule','edm','--sigma_max','10.0',
            '--noise_init','oracle','--noise_model','freq_dep','--beta_noise','5.0',
            '--m_step_mode','clamp','--m_step_start_frac','0.0','--gamma','0.0',
            '--beta_fpdc','1.0','--alpha_ema','0.95',
            '--target_resolution','320','320','--device','cuda',
            '--methods','dps','adps',
            '--output_dir',out]
        r = subprocess.run(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        print(r.stdout[-4000:])
        if r.returncode != 0:
            raise RuntimeError(f'rc={r.returncode} R={R} T={T}')
        print(f'  done in {(time.time()-t0)/60:.1f} min')


In [ ]:
# Cell U2 - Knee single-coil T-sweep on 192 slices, all 4 methods.
# Supplies the knee row of tab:results AND back-fills tab:tscaling_knee
# in case any cells used the 5-slice pilot.
import os, subprocess, time
REPO   = '/content/fastmri'
DATA   = 'data/singlecoil_val'  # adjust if your knee data lives elsewhere
CKPT   = 'checkpoints/edm/supervised_R=1'
NSL    = 192
CF     = {4: 0.08, 8: 0.04}
TAG    = 'fd_b5_smax10_clamp_oracle_v4_knee'
OUTROOT = f'outputs/Tsweep_{TAG}'
os.makedirs(f'{REPO}/{OUTROOT}', exist_ok=True)
for R in (4, 8):
    for T in (10, 20, 50, 100):
        out = f'{OUTROOT}/R{R}_T{T}'
        if os.path.exists(f'{REPO}/{out}/results.json'):
            print('  skip (exists):', out); continue
        print(f'\n===== knee R={R} T={T} -> {out} =====')
        t0 = time.time()
        cmd = ['python','-u','scripts/reconstruct.py',
            '--mode','edm','--checkpoint_dir',CKPT,'--data_path',DATA,
            '--num_slices',str(NSL),'--whole_volume',
            '--acceleration',str(R),'--center_fraction',str(CF[R]),
            '--num_steps',str(T),'--schedule','edm','--sigma_max','10.0',
            '--noise_init','oracle','--noise_model','freq_dep','--beta_noise','5.0',
            '--m_step_mode','clamp','--m_step_start_frac','0.0','--gamma','0.0',
            '--beta_fpdc','1.0','--alpha_ema','0.95',
            '--target_resolution','320','320','--device','cuda',
            '--methods','dps','adps','pigdm','fakgd',
            '--output_dir',out]
        r = subprocess.run(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        print(r.stdout[-4000:])
        if r.returncode != 0:
            raise RuntimeError(f'rc={r.returncode} R={R} T={T}')
        print(f'  done in {(time.time()-t0)/60:.1f} min')


In [ ]:
# Cell U3 - beta x alpha grid for tab:hparam on 192 brain slices, R=4, T=20.
# 12 FA-KGD runs (3 betas x 4 alphas). PiGDM column comes from the existing
# Tsweep_fd_b5_smax10_clamp_oracle_v4/R4_T20/results.json, so this cell only
# needs to run FA-KGD.
import os, subprocess, time, itertools
REPO   = '/content/fastmri'
REPO   = '/content/fastmri'
DATA   = 'data/multicoil_val'
CKPT   = 'checkpoints/edm/supervised_R=1'
NSL    = 12 * 16
TAG    = 'hparam_grid_R4_T20'
OUTROOT = f'outputs/{TAG}'
os.makedirs(f'{REPO}/{OUTROOT}', exist_ok=True)
BETAS  = [0.5, 1.0, 2.0]
ALPHAS = [0.80, 0.90, 0.95, 1.00]
for beta, alpha in itertools.product(BETAS, ALPHAS):
    out = f'{OUTROOT}/b{beta}_a{alpha}'
    if os.path.exists(f'{REPO}/{out}/results.json'):
        print('  skip (exists):', out); continue
    print(f'\n===== beta={beta} alpha={alpha} -> {out} =====')
    t0 = time.time()
    cmd = ['python','-u','scripts/reconstruct.py',
        '--mode','edm','--checkpoint_dir',CKPT,'--data_path',DATA,
        '--num_slices',str(NSL),'--whole_volume',
        '--acceleration','4','--center_fraction','0.08',
        '--num_steps','20','--schedule','edm','--sigma_max','10.0',
        '--noise_init','oracle','--noise_model','freq_dep','--beta_noise','5.0',
        '--m_step_mode','clamp','--m_step_start_frac','0.0','--gamma','0.0',
        '--beta_fpdc',str(beta),'--alpha_ema',str(alpha),
        '--target_resolution','320','320','--device','cuda',
        '--methods','fakgd',
        '--output_dir',out]
    r = subprocess.run(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        raise RuntimeError(f'rc={r.returncode} beta={beta} alpha={alpha}')
    print(f'  done in {(time.time()-t0)/60:.1f} min')


In [ ]:
# Cell U4 - Per-band k-space NMSE on 192 brain R=4 slices for tab:freq.
# reconstruct.py already saves per-slice recons as torch .pt files at
#   {output_dir}/{vol_stem}/slice_{sl:03d}.pt
# with keys pigdm_recon / fakgd_recon / x_gt etc. No patching needed.
import os, subprocess, time, json, glob
import numpy as np
REPO = '/content/fastmri'

# Make sure reconstruct.py is pristine (revert any previous patch attempts).
rc = subprocess.run(['git', 'diff', '--quiet', '--', 'scripts/reconstruct.py'],
                    cwd=REPO)
if rc.returncode != 0:
    subprocess.run(['git', 'checkout', '--', 'scripts/reconstruct.py'],
                   cwd=REPO, check=True)
    print('  reverted modified scripts/reconstruct.py to pristine')

DATA   = 'data/multicoil_val'
CKPT   = 'checkpoints/edm/supervised_R=1'
NSL    = 12 * 16
TAG    = 'freq_R4_T50_recons'
OUT    = f'outputs/{TAG}'
os.makedirs(f'{REPO}/{OUT}', exist_ok=True)

# Force re-run if results.json exists but there are no per-slice .pt files
# (e.g. previous broken-patch run aborted before any slice was saved).
def _has_pt():
    return bool(glob.glob(f'{REPO}/{OUT}/*/slice_*.pt'))

if os.path.exists(f'{REPO}/{OUT}/results.json') and not _has_pt():
    print('  removing stale results.json (no .pt slices on disk)')
    os.remove(f'{REPO}/{OUT}/results.json')

if not os.path.exists(f'{REPO}/{OUT}/results.json'):
    cmd = ['python','-u','scripts/reconstruct.py',
        '--mode','edm','--checkpoint_dir',CKPT,'--data_path',DATA,
        '--num_slices',str(NSL),'--whole_volume',
        '--acceleration','4','--center_fraction','0.08',
        '--num_steps','50','--schedule','edm','--sigma_max','10.0',
        '--noise_init','oracle','--noise_model','freq_dep','--beta_noise','5.0',
        '--m_step_mode','clamp','--m_step_start_frac','0.0','--gamma','0.0',
        '--beta_fpdc','1.0','--alpha_ema','0.95',
        '--target_resolution','320','320','--device','cuda',
        '--methods','pigdm','fakgd',
        '--output_dir',OUT]
    t0 = time.time()
    r = subprocess.run(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        raise RuntimeError(f'rc={r.returncode}')
    print(f'  recon dump done in {(time.time()-t0)/60:.1f} min')
else:
    print('  reusing existing recons in', OUT)

# 2) Score per-band k-space NMSE per slice and average.
import h5py, torch
BANDS = [(0,50),(50,100),(100,150),(150,200),(200,260)]

def radius_grid(H, W):
    yy, xx = np.meshgrid(np.arange(H)-H/2, np.arange(W)-W/2, indexing='ij')
    return np.sqrt(yy**2 + xx**2)

def per_band_nmse(rec, gt):
    H, W = gt.shape
    Krec = np.fft.fftshift(np.fft.fft2(rec))
    Kgt  = np.fft.fftshift(np.fft.fft2(gt))
    r = radius_grid(H, W)
    out = []
    for lo, hi in BANDS:
        m = (r >= lo) & (r < hi)
        denom = (np.abs(Kgt[m])**2).sum()
        num = (np.abs(Krec[m] - Kgt[m])**2).sum()
        out.append(num / max(denom, 1e-12))
    return np.array(out)

pt_files = sorted(glob.glob(f'{REPO}/{OUT}/*/slice_*.pt'))
assert pt_files, f'no slice .pt files under {REPO}/{OUT}'
print(f'  found {len(pt_files)} slice .pt files')

scores_pi, scores_fa = [], []
for pt in pt_files:
    d = torch.load(pt, map_location='cpu', weights_only=False)
    if 'pigdm_recon' not in d or 'fakgd_recon' not in d: continue
    gt = d['x_gt'].abs().numpy()
    pi = d['pigdm_recon'].abs().numpy()
    fa = d['fakgd_recon'].abs().numpy()
    while gt.ndim > 2: gt = gt.squeeze(0)
    while pi.ndim > 2: pi = pi.squeeze(0)
    while fa.ndim > 2: fa = fa.squeeze(0)
    # Center-crop pi/fa to gt shape if larger.
    H, W = gt.shape
    if pi.shape != gt.shape:
        h0 = (pi.shape[0]-H)//2; w0 = (pi.shape[1]-W)//2
        pi = pi[h0:h0+H, w0:w0+W]
    if fa.shape != gt.shape:
        h0 = (fa.shape[0]-H)//2; w0 = (fa.shape[1]-W)//2
        fa = fa[h0:h0+H, w0:w0+W]
    scores_pi.append(per_band_nmse(pi, gt))
    scores_fa.append(per_band_nmse(fa, gt))

print(f'  scored {len(scores_pi)} slices')
scores_pi = np.stack(scores_pi).mean(0)
scores_fa = np.stack(scores_fa).mean(0)
rel = (scores_pi - scores_fa) / scores_pi * 100.0
print(f"\n{'band':>10} {'PiGDM':>10} {'FA-KGD':>10} {'Delta %':>10}")
for (lo,hi), p, f, r_ in zip(BANDS, scores_pi, scores_fa, rel):
    print(f"{lo:>4}-{hi:<5} {p:>10.4f} {f:>10.4f} {r_:>9.2f}%")


In [ ]:
# === NUMPY_REPAIR === fix scikit-image/numpy ABI mismatch from Cell 3.
# If numpy was downgraded below scikit-image's build ABI, reinstall a compatible
# numpy in-place. This is a no-op if numpy is already OK.
import subprocess, sys
try:
    from skimage.metrics import structural_similarity  # probe
except ImportError:
    print('  repairing numpy/scikit-image ABI...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           '--force-reinstall', '--no-deps', 'numpy>=2.3'])
    # Hard kernel restart so the fresh numpy is loaded. Colab auto-reconnects;
    # re-run U5 after reconnect (skip Cells 1-4, filesystem is preserved).
    import os
    print('  restarting kernel - re-run U5 after reconnect')
    os.kill(os.getpid(), 9)

# Cell U5 - VarNet brain re-eval across all 12 validation volumes (no training,
# inference only). Supplies the supervised-reference row of tab:results at the
# same volume coverage as PiGDM/FA-KGD.
import os, sys, glob, json, time
import numpy as np, h5py, torch
REPO = '/content/fastmri'
sys.path.insert(0, REPO)
from skimage.metrics import structural_similarity as ssim_fn
from scripts.run_varnet_baseline import run_varnet_on_brain, create_equispaced_mask
from fastmri.models import VarNet

model = VarNet(num_cascades=12, pools=4, chans=18, sens_pools=4, sens_chans=8)
sd = torch.load(f'{REPO}/checkpoints/varnet/brain_leaderboard_state_dict.pt',
                map_location='cpu', weights_only=False)
model.load_state_dict(sd); model.eval()

VOLS = sorted(glob.glob(f'{REPO}/data/multicoil_val/file_brain_AXT2_*.h5'))[:12]
SLICES_PER_VOL = list(range(0, 16))   # 12 x 16 = 192

all_results = {'R4': {}, 'R8': {}}
for R in (4, 8):
    print(f'\n=== VarNet brain R={R} ===')
    for vp in VOLS:
        print(' vol:', os.path.basename(vp))
        all_results[f'R{R}'][os.path.basename(vp)] = run_varnet_on_brain(
            model, vp, SLICES_PER_VOL, acceleration=R)

OUT = f'{REPO}/outputs/varnet_baseline_12vol.json'
with open(OUT, 'w') as f:
    json.dump(all_results, f, indent=2)
print('saved ->', OUT)

# Aggregate
for R in (4, 8):
    psnrs = [s['psnr'] for v in all_results[f'R{R}'].values() for s in v]
    ssims = [s['ssim'] for v in all_results[f'R{R}'].values() for s in v]
    print(f' R={R}: n={len(psnrs)} PSNR={np.mean(psnrs):.2f}+-{np.std(psnrs):.2f} '
          f'SSIM={np.mean(ssims):.4f}+-{np.std(ssims):.4f}')


In [ ]:
# Cell U6 - Aggregate everything that was just produced and print
# paste-ready table fragments for the paper.
import os, glob, json, numpy as np
REPO = '/content/fastmri'

def mean_psnr(results_json, key):
    if not os.path.exists(results_json): return None
    d = json.load(open(results_json))
    pv = d.get('per_volume') or {}
    if pv:
        return float(np.mean([v[key] for v in pv.values() if key in v]))
    ps = d.get('per_slice') or []
    k = key.replace('_psnr','') + '_psnr'
    return float(np.mean([s[k] for s in ps if k in s and s[k] == s[k]]))

# tab:scaling DPS column (brain R=4 T in {20,50,100})
print('== tab:scaling DPS column ==')
DSADPS = f'{REPO}/outputs/Tsweep_fd_b5_smax10_clamp_oracle_v4_dpsadps'
for T in (20, 50, 100):
    p = f'{DSADPS}/R4_T{T}/results.json'
    print(f'  T={T:3d}  DPS={mean_psnr(p,"dps_psnr")}  ADPS={mean_psnr(p,"adps_psnr")}')

print('\n== tab:results candidate cells ==')
# brain R=4 T=50 and R=8 T=50
PI = f'{REPO}/outputs/Tsweep_fd_b5_smax10_clamp_oracle_v4'
for R, T in [(4, 50), (8, 50)]:
    p_pi = f'{PI}/R{R}_T{T}/results.json'
    p_da = f'{DSADPS}/R{R}_T{T}/results.json'
    print(f'  brain R={R} T={T}: PiGDM={mean_psnr(p_pi,"pigdm_psnr")} '
          f'FA-KGD={mean_psnr(p_pi,"fakgd_psnr")} '
          f'DPS={mean_psnr(p_da,"dps_psnr")} ADPS={mean_psnr(p_da,"adps_psnr")}')
# knee
KN = f'{REPO}/outputs/Tsweep_fd_b5_smax10_clamp_oracle_v4_knee'
for R, T in [(4, 20), (8, 20)]:
    p = f'{KN}/R{R}_T{T}/results.json'
    print(f'  knee R={R} T={T}: PiGDM={mean_psnr(p,"pigdm_psnr")} '
          f'FA-KGD={mean_psnr(p,"fakgd_psnr")} '
          f'DPS={mean_psnr(p,"dps_psnr")} ADPS={mean_psnr(p,"adps_psnr")}')

print('\n== tab:hparam grid (Delta PSNR over PiGDM, brain R=4 T=20) ==')
PI_BASE = mean_psnr(f'{PI}/R4_T20/results.json', 'pigdm_psnr')
print(f'  PiGDM baseline = {PI_BASE:.4f}')
for beta in (0.5, 1.0, 2.0):
    row = []
    for alpha in (0.80, 0.90, 0.95, 1.00):
        p = f'{REPO}/outputs/hparam_grid_R4_T20/b{beta}_a{alpha}/results.json'
        fa = mean_psnr(p, 'fakgd_psnr')
        row.append(f'{fa-PI_BASE:+.3f}' if fa is not None and PI_BASE is not None else '   -  ')
    print(f'  beta={beta}: {" ".join(row)}')

print('\n== VarNet 12-vol summary ==')
V = json.load(open(f'{REPO}/outputs/varnet_baseline_12vol.json')) if os.path.exists(f'{REPO}/outputs/varnet_baseline_12vol.json') else {}
for R in ('R4', 'R8'):
    if R in V:
        psnrs = [s['psnr'] for v in V[R].values() for s in v]
        print(f'  {R}: PSNR={np.mean(psnrs):.2f}+-{np.std(psnrs):.2f}  n={len(psnrs)}')
